# Week 3 - Dataset Audit

專題：PQC × IoT / Embedded / Product Security

這個 Notebook 主要是實際檢查目前 Dataset 的 shape、missing、duplicates、label 分布、hardware coverage、provenance 和 split 是否可行。

In [1]:
from pathlib import Path
import pandas as pd

base = Path("..")
scenario = pd.read_csv(base / "data" / "pqc_scenario_dataset_v0.1.csv")
evidence = pd.read_csv(base / "data" / "pqc_evidence_corpus_v0.1.csv")

print("Scenario Dataset:", scenario.shape)
print("Evidence Corpus:", evidence.shape)


Scenario Dataset: (12, 17)
Evidence Corpus: (18, 30)


## 1. Missing Values

In [2]:
pd.DataFrame({
    "missing_count": scenario.isna().sum(),
    "missing_rate_pct": (scenario.isna().mean() * 100).round(1)
}).sort_values("missing_rate_pct", ascending=False)


,missing_count,missing_rate_pct
bandwidth_limit_kbps,12,100.0
power_limit_mw,12,100.0
threat_model,12,100.0
max_latency_ms,12,100.0
scenario_id,0,0.0
available_flash_bytes,0,0.0
device_platform,0,0.0
available_ram_bytes,0,0.0
cpu_architecture,0,0.0
application_scenario,0,0.0


In [3]:
pd.DataFrame({
    "missing_count": evidence.isna().sum(),
    "missing_rate_pct": (evidence.isna().mean() * 100).round(1)
}).sort_values("missing_rate_pct", ascending=False).head(15)


,missing_count,missing_rate_pct
record_id,0,0.0
algorithm_family,0,0.0
parameter_set,0,0.0
crypto_function,0,0.0
nist_standard,0,0.0
security_category,0,0.0
implementation,0,0.0
cpu_architecture,0,0.0
benchmark_platform_scope,0,0.0
benchmark_frequency_mhz,0,0.0


## 2. Duplicate Check

In [4]:
print("Scenario exact duplicates:", scenario.duplicated().sum())
print("Evidence exact duplicates:", evidence.duplicated().sum())
print("Unique scenario_id:", scenario["scenario_id"].nunique(), "/", len(scenario))
print("Unique record_id:", evidence["record_id"].nunique(), "/", len(evidence))


Scenario exact duplicates: 0
Evidence exact duplicates: 0
Unique scenario_id: 12 / 12
Unique record_id: 18 / 18


## 3. Label and Function Distribution

In [5]:
print("Crypto Function")
print(scenario["crypto_function"].value_counts())
print("\nRequired Security Category")
print(scenario["required_security_category"].value_counts().sort_index())
print("\nLabel Status")
print(scenario["label_status"].value_counts())


Crypto Function
crypto_function
KEM                  6
Digital Signature    6
Name: count, dtype: int64

Required Security Category
required_security_category
1    2
2    2
3    4
5    4
Name: count, dtype: int64

Label Status
label_status
needs_human_review    12
Name: count, dtype: int64


## 4. Hardware Coverage and Group IDs

In [6]:
print("Device Platform")
print(scenario["device_platform"].value_counts())
print("\nCPU Architecture")
print(scenario["cpu_architecture"].value_counts())
print("\nUnique platform groups:", scenario["device_platform"].nunique())


Device Platform
device_platform
NUCLEO-L4R5ZI        6
STM32F4 Discovery    6
Name: count, dtype: int64

CPU Architecture
cpu_architecture
ARM Cortex-M4    12
Name: count, dtype: int64

Unique platform groups: 2


## 5. Evidence Coverage

In [7]:
print("Algorithm Family")
print(evidence["algorithm_family"].value_counts())
print("\nParameter Set")
print(evidence["parameter_set"].value_counts())
print("\nImplementation")
print(evidence["implementation"].value_counts())


Algorithm Family
algorithm_family
ML-KEM    9
ML-DSA    9
Name: count, dtype: int64

Parameter Set
parameter_set
ML-KEM-512     3
ML-KEM-768     3
ML-KEM-1024    3
ML-DSA-44      3
ML-DSA-65      3
ML-DSA-87      3
Name: count, dtype: int64

Implementation
implementation
clean       6
m4fstack    6
m4fspeed    3
m4f         3
Name: count, dtype: int64


## 6. Time / Provenance

In [8]:
print("Retrieved date range:", evidence["retrieved_date"].min(), "to", evidence["retrieved_date"].max())
print("Benchmark blob SHA:", evidence["benchmark_file_blob_sha"].drop_duplicates().tolist())
print("Current limitation: original benchmark publication/commit time is not stored per row.")


Retrieved date range: 2026-09-29 to 2026-09-29
Benchmark blob SHA: ['9a62205f01bb3c848d7bba037c2011a8eadbd0d4']
Current limitation: original benchmark publication/commit time is not stored per row.


## 7. Split Feasibility

In [9]:
group_counts = scenario.groupby("device_platform").size()
print(group_counts)

if scenario["device_platform"].nunique() < 3:
    print("\nFinding: fewer than 3 hardware groups.")
    print("A non-overlapping Train/Validation/Test Group Split is not currently feasible.")
else:
    print("\nAt least 3 groups are available for a 3-way Group Split.")


device_platform
NUCLEO-L4R5ZI        6
STM32F4 Discovery    6
dtype: int64

Finding: fewer than 3 hardware groups.
A non-overlapping Train/Validation/Test Group Split is not currently feasible.


## Audit Decision

這次 Audit 後，我目前的判斷是：

- 最後仍希望使用 Group Split，因為 Test 要模擬新的 hardware platform
- 目前只有兩個 platform group，所以暫時不適合硬切 Train / Validation / Test 三組
- 現在所有 Label 都還是 weak label，需要再人工確認
- bandwidth、latency、power 和 threat model 目前都是 100% missing
- 現有資料全部是 ARM Cortex-M4，所以不能直接代表所有 IoT / Embedded 平台